# Heart Disease UCI — Exploratory Data Analysis

**Task 1 of the MLOps assignment.** This notebook loads the UCI Heart Disease (Cleveland) dataset, inspects data quality, and produces the required professional visualisations: histograms, correlation heatmap, class-balance and feature-relationship plots.

The reusable logic lives in `src/` so the exact same steps run headlessly in CI (`python -m src.data.eda`). This notebook is the narrated version.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))  # so `import src...` works from notebooks/

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='deep')

from src.data.download import download
from src.data.preprocess import clean
from src.config import RAW_CSV, NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET

## 1. Acquire the data
The `download()` helper fetches the dataset via the official `ucimlrepo` package (id=45), falling back to a direct download of `processed.cleveland.data` from the UCI archive. It writes a headered CSV to `data/raw/`.

In [ ]:
raw = download()
print('Shape:', raw.shape)
raw.head()

## 2. Missing-value analysis
The Cleveland file uses `?` for missing entries (already converted to `NaN` on download). Only `ca` and `thal` contain gaps — a handful of rows — which we impute inside the modelling pipeline rather than dropping.

In [ ]:
missing = raw.isna().sum()
print(missing[missing > 0])

## 3. Clean & binarise the target
`clean()` coerces every column to numeric, binarises the 0–4 severity `num` column into a 0/1 `target` (0 = no disease, ≥1 = disease), and drops exact duplicates.

In [ ]:
df = clean(raw)
print('Clean shape:', df.shape)
print('\nClass balance:')
print(df[TARGET].value_counts(normalize=True).round(3))
df.describe().T

## 4. Class balance
The two classes are close to balanced (~46% positive), so accuracy is a reasonable headline metric, though we still report precision/recall/F1/ROC-AUC.

In [ ]:
ax = df[TARGET].map({0:'No disease',1:'Disease'}).value_counts().plot(kind='bar', color=['#4c72b0','#dd8452'])
ax.set_title('Class Balance'); ax.set_ylabel('Count'); plt.xticks(rotation=0); plt.show()

## 5. Distributions of numeric features

In [ ]:
df[NUMERIC_FEATURES].hist(figsize=(12,8), bins=20, color='#4c72b0', edgecolor='white')
plt.suptitle('Numeric Feature Distributions'); plt.tight_layout(); plt.show()

## 6. Correlation heatmap
`cp` (chest pain type), `thalach` (max heart rate), `oldpeak`, `ca` and `exang` show the strongest correlations with the target — clinically sensible risk indicators.

In [ ]:
plt.figure(figsize=(11,9))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, annot_kws={'size':7})
plt.title('Feature Correlation Heatmap'); plt.tight_layout(); plt.show()

## 7. Feature relationships with the target
Patients with heart disease tend to have lower max heart rate (`thalach`), higher `oldpeak`, and more affected vessels (`ca`).

In [ ]:
fig, axes = plt.subplots(1,4, figsize=(16,4))
label = df[TARGET].map({0:'No disease',1:'Disease'})
for ax, feat in zip(axes, ['age','thalach','oldpeak','chol']):
    sns.boxplot(x=label, y=df[feat], ax=ax, hue=label, legend=False)
    ax.set_title(f'{feat} by target'); ax.set_xlabel('')
plt.tight_layout(); plt.show()

## Summary
- 303 patients, 13 features, binary target, ~46% positive.
- Missing values only in `ca` (4) and `thal` (2) → imputed in-pipeline.
- Strongest target correlates: `cp`, `thalach`, `oldpeak`, `ca`, `exang`.
- Next: `src/models/train.py` builds the leak-free `ColumnTransformer` + model pipeline, tunes with `GridSearchCV`, and tracks everything in MLflow.